In [ ]:
import pandas as pd
import requests
import time
import os

In [ ]:
INPUT_CSV  = "dataset/all_multi_paragraphs_2022_2026.csv"
OUTPUT_CSV = "dataset/all_multi_paragraphs_2022_2026_translated.csv"
MODEL      = "mistral-nemo"
OLLAMA_URL = "http://localhost:11434/api/generate"
TEXT_COL   = "text_block"

START_ROW  = 0      # ← change this each run
STOP_ROW   = 50     # ← change this each run (50 for testing)

In [ ]:
PROMPT_TEMPLATE = """You are a translator for news articles. You will receive a multiparagraph in this form:

Text: <multi-paragraph>

Reminder: Preserve names, numbers, and quotes exactly. Return only the translation. Do not comment on the translation.

You are given a multi-paragraph from a news article and translate it into clear English. Preserve names, numbers, and quotes exactly. Return only the translation. Do not comment on the translation.

If paragraphs contain sentence fragments, assume it is a headline or image description. Translate them as sentence fragments as well. Maintain the paragraph structure.

Some entity names will be masked as [Gruppe 1] and [andere Gruppe]. Maintain those names.

Output the translation as:

Text: <translated text>

Text: {text}"""

def translate(text, retries=3):
    for attempt in range(retries):
        try:
            resp = requests.post(OLLAMA_URL, json={
                "model": MODEL,
                "prompt": PROMPT_TEMPLATE.format(text=text),
                "stream": False
            }, timeout=120)
            raw = resp.json()["response"].strip()
            if raw.startswith("Text:"):
                raw = raw[len("Text:"):].strip()
            return raw
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5)
    return None

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows total")

chunk = df.iloc[START_ROW:STOP_ROW].copy().reset_index(drop=True)
print(f"Processing rows {START_ROW} → {STOP_ROW} ({len(chunk)} rows)")

# Fix empty output file + set local_start
if os.path.exists(OUTPUT_CSV):
    if os.path.getsize(OUTPUT_CSV) == 0:
        os.remove(OUTPUT_CSV)
        print("Removed empty output file, starting fresh")
        local_start = 0
    else:
        done = pd.read_csv(OUTPUT_CSV)
        local_start = max(0, len(done) - START_ROW)
        print(f"Found {len(done)} already translated rows, resuming from local row {local_start}")
else:
    local_start = 0
    print("Starting fresh")

for i in range(local_start, len(chunk)):
    text = chunk.at[i, TEXT_COL]
    if pd.isna(text) or str(text).strip() == "":
        chunk.at[i, TEXT_COL + "_en"] = ""
    else:
        chunk.at[i, TEXT_COL + "_en"] = translate(str(text))

    # Save every row immediately
    row_out = chunk.iloc[[i]]
    write_header = not os.path.exists(OUTPUT_CSV)
    row_out.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)

    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(chunk)} done (global row {START_ROW + i + 1})")

print(f"\n✓ Done! Rows {START_ROW}–{STOP_ROW} saved to {OUTPUT_CSV}")

Loaded 659895 rows total
Processing rows 0 → 50 (50 rows)
Starting fresh
  [attempt 1] Error: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=120)


KeyboardInterrupt: 

In [ ]:
result = pd.read_csv(OUTPUT_CSV)
print(f"Translated rows so far: {len(result)}")
result[[TEXT_COL, TEXT_COL + "_en"]].head(10)